# NB05a Quantification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB05a_Quantification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB05a/NB05a_Quantification.ipynb)

1. Quantification Theory
    1. Targeted quantification
    2. (i) Targeted acquisition at peptide
    3. (ii) Targeted data analysis at peptide ion level
2. References

## Quantification Theory

To estimate the amount of individual proteins in complex mixtures, all peptide signals corresponding to a common protein serve as a 
proxy for their abundance. Peptide information needs to be obtained from multidimensional signal data detected by the mass spectrometer. 
All signals generated from one peptide ion species, often referred to as peptide feature, need to be grouped to form a three-dimensional peak 
along the m/z, ion intensity, and retention time dimension. This process is generally defined as peak detection or feature detection. 
Peak detection algorithms are a set of rules defining how neighboring signal points are joined. Whether noise filtering is done before or after 
peak detection strongly depends on the peak detection algorithm. Traditional approaches mainly focused on signal amplitude neglecting 
characteristic peak shapes as a common feature of chromatographic or spectroscopic peaks. These algorithms are prone to miss detection of low 
intensity peaks with a signal strength close to the noise level. To overcome these issues, techniques like smoothing, shape-matching and curve 
fitting are often implemented and applied. At the time, the most promising approach to do shape-matching and noise reduction in one step uses the 
continuous wavelet transformation (CWT).

In general, a CWT based approach describes a family of time-frequency-transformations often used in data compression and feature detection. 
The term is coined by the use of a wavelet, as a basis function which is “compared” to the signal. The point of highest correlation between the 
basis function and the signal reflects the location of the peak present. Due to the fact that MS derived peaks often follow the shape of a 
gaussian distribution, the *Mexican Hat* wavelet as the negative normalized second derivative of the Gaussian distribution is perfectly 
suited to find the peptide feature.

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/Wavelets.png)

**Figure 5: Schematic representation of the ‘Haar’-wavelet (blue) and the ‘Mexican Hat’- wavelet (green). **
The ‘Haar’-wavelet is named after its discoverer Alfred Haar and represents the first wavelet ever to be described. The ‘Mexican Hat’- or ‘Ricker’-wavelet is 
frequently used in the fields of signal detection and compression.

Depending on the quantification approach, the peptide features used for protein quantification might differ. In case of isotopic labeling, 
quantification means pairing features with the proper mass shift according to the utilized label. It is essential to account for the frequency 
of label incorporation when calculating the mass shift for the utilized label. Taking the ICAT method as an example, by which a heavy/light 
difference of 9 Dalton per cysteine is incorporated, the total mass shift is 9 Dalton times the number of cysteine within the sequence. 
Consequently, pairing peptide features for 15N labeling is even more challenging, as the mass shift is less discrete. Using stable 
isotope labeling, different peptide feature pairs belonging to the same protein can be treated as technical replicates and averaged to gain 
protein quantification. In contrast, the sum of all extracted peptide signals results in a label-free protein quantiﬁcation. Spectral counting 
computes abundance values from the number of times a peptide was successfully identiﬁed by tandem mass spectrometry (MS/MS) and combines all 
these events per protein. The spectral counting values can be normalized by the number of peptides theoretically expected from the particular 
protein. 

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinQuantification.png)

**Figure 6: Computational strategy of peptide and protein quantiﬁcation on based on stable isotope labeling or by label-free quantiﬁcation.**
(A) Label-free methods compare corresponding peptide abundances over different MS runs. The abundance is either 
estimated by the elution proﬁle les of the pep de ions or (B) in case of spectral counting, by the number of times a peptide was 
successfully identiﬁed (MS2). In contrast, methods based on differential stable isotope labeling analyze peptides pairs detected by 
their characteristic mass diﬀerence Δm/z. The abundance is estimated by the ratio of their corresponding elution proﬁles (C). Isobaric 
tagging methods (D) compare the reporter ion abundances in the fragmentation spectrum.
### Targeted quantification

Targeted proteomics has gained significant popularity in mass spectrometry‐based protein quantification as a method to detect proteins of 
interest with high sensitivity, quantitative accuracy and reproducibility. The two major strategies of (i) targeted acquisition at peptide, 
and (ii) targeted data analysis at peptide ion level need to be distinguished.
###(i) Targeted acquisition at peptide

In multiple reaction monitoring (MRM or SRM for single/selected reaction monitoring) simply predefined transitions are recorded. 
Knowledge about the targeted transitions from precursor to their corresponding fragment ions are needed and predefined in the mass 
spectrometer. MRM can be performed rapidly and is highly specific even for low abundant peptide ions in complex mixtures, but with the 
drawback of a necessary bias in the sense that only predefined peptides are measured.
### (ii) Targeted data analysis at peptide ion level

Data‐independent acquisition at the peptide level makes it possible to acquire peptide data for virtually all peptide ions present in a sample. 
In this strategy, a high‐resolution mass analyzer—such as an orbitrap or a time‐of‐flight—is used to constantly sample the full mass range 
at the peptide level during the entire chromatographic gradient. In a subsequent step, precursor ion chromatograms can be extracted by targeted 
data analysis. Those extracted-ion chromatogram (XIC) can be obtained to calculate the area under the curve and used for peptide quantification.

Let’s start and extract a XIC…


In [1]:
#r "nuget: FSharp.Stats, 0.4.3"
#r "nuget: BioFSharp, 2.0.0-beta5"
#r "nuget: BioFSharp.IO, 2.0.0-beta5"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: System.Data.SQLite, 1.0.113.7"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.1"
#r "nuget: MzIO.SQL, 0.1.4"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: Deedle, 3.0.0"
#r "nuget: Deedle.Interactive, 3.0.0"

open Plotly.NET
open FSharp.Stats
open BioFSharp
open System.IO
open System.Data.SQLite
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL
open Deedle
open Deedle.Interactive


Installed Packages BioFSharp, 2.0.0-beta5 BioFSharp.IO, 2.0.0-beta5 BioFSharp.Mz, 0.1.5-beta Deedle, 3.0.0 Deedle.Interactive, 3.0.0 FSharp.Stats, 0.4.3 MzIO, 0.1.1 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.4 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0 System.Data.SQLite, 1.0.113.7

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

Loading extensions from `/home/paulinehans/.nuget/packages/deedle.interactive/3.0.0/lib/netstandard2.1/Deedle.Interactive.dll`

In the last Notebook we get at the End our Scoring.tsv file. The first step is now, to load this file and get accsess to it.

In [2]:
// Code-Block 1id

let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"/home/paulinehans/Dokumente/scoring.tsv"|]

let data = Deedle.Frame.ReadCsv (path, hasHeaders =  true,  separators = "\t")
data

0,->,2,1295.75613484367,0.7645020568323313,LQNIVGVPTSIR,63.808983333333,sample=1 period=1 cycle=3887 experiment=5
1,->,2,1458.7255629902602,0,LIFQYASFNNSR,63.784983333333,sample=1 period=1 cycle=3886 experiment=12
2,->,2,1362.6626879473802,1.169364996777007,GILASDESNATTGK,63.78165,sample=1 period=1 cycle=3886 experiment=10
3,->,2,920.49673227576,0.3234160188668125,FIESQVAK,63.768316666667,sample=1 period=1 cycle=3886 experiment=2
4,->,2,1502.85691151298,0,SVVSIPHGPSIIAAR,63.75015,sample=1 period=1 cycle=3885 experiment=16
:,,...,...,...,...,...,...
3817,->,2,941.50830002009,0,SVLPANWR,7.608533333333,sample=1 period=1 cycle=1327 experiment=2
3818,->,2,917.45665836781,0.5472311581473039,SNSTPLGSR,7.46355,sample=1 period=1 cycle=1307 experiment=2
3819,->,2,917.45665836781,0,SNSTPLGSR,7.293733333333,sample=1 period=1 cycle=1282 experiment=2
3820,->,2,933.49198124849,1.1403710071243618,LVAFDNQK,7.206233333333,sample=1 period=1 cycle=1269 experiment=2
3821,->,2,917.45665836781,0,SNSTPLGSR,7.036066666667,sample=1 period=1 cycle=1244 experiment=2


We have now a Dataframe, looks like a table, works like a table. What we do now is relative Quantification in the form of XICs. For that we need several types of information: 
- precursor mass
- charge 
- peptide sequence
- retention time
- labeled data (15N)

(and the retention time index but later more to that). The Information is stored in the Dataframe, so the goal is to extract the data out of it so that we can work with it. 

In [3]:
let extract = 
    data 
    |> Frame.sliceCols([|"Mass"; "Charge"; "PeptideSequence"; "RetentionTime"; "MatchingScore"|])
extract

0,->,1295.75613484367,2,LQNIVGVPTSIR,63.808983333333,0.7645020568323313
1,->,1458.7255629902602,2,LIFQYASFNNSR,63.784983333333,0
2,->,1362.6626879473802,2,GILASDESNATTGK,63.78165,1.169364996777007
3,->,920.49673227576,2,FIESQVAK,63.768316666667,0.3234160188668125
4,->,1502.85691151298,2,SVVSIPHGPSIIAAR,63.75015,0
:,,...,...,...,...,...
3817,->,941.50830002009,2,SVLPANWR,7.608533333333,0
3818,->,917.45665836781,2,SNSTPLGSR,7.46355,0.5472311581473039
3819,->,917.45665836781,2,SNSTPLGSR,7.293733333333,0
3820,->,933.49198124849,2,LVAFDNQK,7.206233333333,1.1403710071243618
3821,->,917.45665836781,2,SNSTPLGSR,7.036066666667,0


Nice! We have now our columns of interest. Now we will get the values out of the columns

In [4]:

let precursorMass = extract.GetColumn<float>("Mass").Values |> Seq.toArray
let rt   = extract.GetColumn<float>("RetentionTime").Values |> Seq.toArray
let charge = extract.GetColumn<int>("Charge").Values |> Seq.toArray
let sequence = extract.GetColumn<string>("PeptideSequence").Values |> Seq.toArray
let score = extract.GetColumn<float>("MatchingScore").Values |> Seq.toArray


The retention-time index is derived from MS1 spectra and is essential because our XIC is built from MS2 information; to align MS2 data with the corresponding MS1 retention-time values, we need this retention-time index. Logically we take our .mzlite file from the last notebook and extract the MS1RTI.

In [5]:
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"downloads/WC_Gr3_UV64 2.mzlite"|]

let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let transaction = mzReader.BeginTransaction()

// Indexes all spectra of the related sample run
let idx = MzIO.Processing.Query.getMS1RTIdx mzReader runID
idx 

let rtIndex = idx |> Seq.toArray |> Array.rev
rtIndex

index value 0 [rt=65.026933333333, spectrumID='sample=1 period=1 cycle=4065 experiment=1'] Rt 65.026933333333 SpectrumID sample=1 period=1 cycle=4065 experiment=1 1 [rt=65.02195, spectrumID='sample=1 period=1 cycle=4064 experiment=1'] Rt 65.02195 SpectrumID sample=1 period=1 cycle=4064 experiment=1 2 [rt=65.016833333333, spectrumID='sample=1 period=1 cycle=4063 experiment=1'] Rt 65.016833333333 SpectrumID sample=1 period=1 cycle=4063 experiment=1 3 [rt=65.011833333333, spectrumID='sample=1 period=1 cycle=4062 experiment=1'] Rt 65.011833333333 SpectrumID sample=1 period=1 cycle=4062 experiment=1 4 [rt=65.006816666667, spectrumID='sample=1 period=1 cycle=4061 experiment=1'] Rt 65.006816666667 SpectrumID sample=1 period=1 cycle=4061 experiment=1 5 [rt=65.001716666667, spectrumID='sample=1 period=1 cycle=4060 experiment=1'] Rt 65.001716666667 SpectrumID sample=1 period=1 cycle=4060 experiment=1 6 [rt=64.996716666667, spectrumID='sample=1 period=1 cycle=4059 experiment=1'] Rt 64.996716666667 SpectrumID sample=1 period=1 cycle=4059 experiment=1 7 [rt=64.991733333333, spectrumID='sample=1 period=1 cycle=4058 experiment=1'] Rt 64.991733333333 SpectrumID sample=1 period=1 cycle=4058 experiment=1 8 [rt=64.986633333333, spectrumID='sample=1 period=1 cycle=4057 experiment=1'] Rt 64.986633333333 SpectrumID sample=1 period=1 cycle=4057 experiment=1 9 [rt=64.981616666667, spectrumID='sample=1 period=1 cycle=4056 experiment=1'] Rt 64.981616666667 SpectrumID sample=1 period=1 cycle=4056 experiment=1 10 [rt=64.976616666667, spectrumID='sample=1 period=1 cycle=4055 experiment=1'] Rt 64.976616666667 SpectrumID sample=1 period=1 cycle=4055 experiment=1 11 [rt=64.9715, spectrumID='sample=1 period=1 cycle=4054 experiment=1'] Rt 64.9715 SpectrumID sample=1 period=1 cycle=4054 experiment=1 12 [rt=64.9665, spectrumID='sample=1 period=1 cycle=4053 experiment=1'] Rt 64.9665 SpectrumID sample=1 period=1 cycle=4053 experiment=1 13 [rt=64.9615, spectrumID='sample=1 period=1 cycle=4052 experiment=1'] Rt 64.9615 SpectrumID sample=1 period=1 cycle=4052 experiment=1 14 [rt=64.956383333333, spectrumID='sample=1 period=1 cycle=4051 experiment=1'] Rt 64.956383333333 SpectrumID sample=1 period=1 cycle=4051 experiment=1 15 [rt=64.951383333333, spectrumID='sample=1 period=1 cycle=4050 experiment=1'] Rt 64.951383333333 SpectrumID sample=1 period=1 cycle=4050 experiment=1 16 [rt=64.946366666667, spectrumID='sample=1 period=1 cycle=4049 experiment=1'] Rt 64.946366666667 SpectrumID sample=1 period=1 cycle=4049 experiment=1 17 [rt=64.94125, spectrumID='sample=1 period=1 cycle=4048 experiment=1'] Rt 64.94125 SpectrumID sample=1 period=1 cycle=4048 experiment=1 18 [rt=64.93625, spectrumID='sample=1 period=1 cycle=4047 experiment=1'] Rt 64.93625 SpectrumID sample=1 period=1 cycle=4047 experiment=1 19 [rt=64.93125, spectrumID='sample=1 period=1 cycle=4046 experiment=1'] Rt 64.93125 SpectrumID sample=1 period=1 cycle=4046 experiment=1 (4045 more)

We're almost done with the data we need. The last puzzle piece is 15N labled data, we're creating this exactly like in the "Isotopic Distribution Notebook"

In [6]:
// Converts a peptide BioSeq into a molecular formula used for mass calculations.
let toFormula bioseq =  
    bioseq
    |> BioSeq.toFormula
    // peptides are hydrolysed in the mass spectrometer, so we add H2O
    |> Formula.add Formula.Table.H2O

// Calculates the monoisotopic mass of a fully 15N-labeled peptide sequence.
let label formula =
    let forumlaBioItem = 
        formula
        // Convert amino-acid string (e.g. "PEPTIDE") to BioSeq.
        |> BioSeq.ofAminoAcidString
        // Build the peptide formula (including added H2O).
        |> toFormula
    // Replace all natural nitrogen (14N) with heavy nitrogen (15N).
    Formula.replaceElement
        forumlaBioItem
        Elements.Table.N
        Elements.Table.Heavy.N15
    // Return monoisotopic mass of the labeled formula.
    |> Formula.monoisoMass

### Simply lovely 
Now we can start with the realtive Quantification! 

The first thing we do is group our information, because every peptide with the same charge and sequence will grouped because the same sequence can be “hit” multiple times in a single run. After that we take the best score (remember here the last task in Peptide Identification Notebook) within a group. For a better handling we create a record type, which stores all our Information (like a box that you can carry around). 

In [7]:
// Record type with all the needed Information
type ConsensLookup = {
    PeptideSequence: string
    RetentionTime: float
    Charge: int
    LabeledMass: float
    PrecursorMass: float
}

// Build one row per index by combining values from all parallel arrays.
let rows = [|
        for i = 0 to rt.Length - 1 do
            // Read sequence and charge at the current index.
            let s = sequence.[i]
            let c = charge.[i]
            // Compute theoretical labeled m/z from sequence and charge.
            let lM = BioFSharp.Mass.toMZ (s |> label) c
            // Store all relevant values as one tuple.
            s, rt.[i], c, lM, precursorMass.[i], score.[i]
    |]
// Build a consensus lookup: one best entry per (sequence, charge) pair.
let consensLookup =
    rows
    // Group rows by peptide sequence and charge state.
    |> Array.groupBy (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> sequence, charge)
    |> Array.map (fun ((sequence,charge), grouped) ->
        // Keep the row with the highest score in each group.
        grouped
        |> Array.maxBy (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> score)
        // Convert the selected tuple into a typed record.
        |> (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> 
            {
                PeptideSequence = sequence
                RetentionTime = rt
                Charge = charge
                LabeledMass = labeledTheoMass
                PrecursorMass = precMass
            }
        )
        //Backup (pls ignore)
        // let medianLabeledTheoMass =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> labeledTheoMass)
        //     |> Array.median
        // let medianRT =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> rt)
        //     |> Array.median
        // let medianPrecMass =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> precMass)
        //     |> Array.median

        // {
        //     PeptideSequence = sequence
        //     RetentionTime = medianRT
        //     Charge = charge
        //     LabeledMass = medianLabeledTheoMass
        //     PrecursorMass = medianPrecMass
        // }
    )
consensLookup

index value 0 { PeptideSequence = "LQNIVGVPTSIR"\n RetentionTime = 42.4451\n Charge = 2\n LabeledMass = 657.3601405\n PrecursorMass = 1295.756135 } PeptideSequence LQNIVGVPTSIR RetentionTime 42.4451 Charge 2 LabeledMass 657.360140482545 PrecursorMass 1295.75613484367 1 { PeptideSequence = "LIFQYASFNNSR"\n RetentionTime = 48.09926667\n Charge = 2\n LabeledMass = 739.343372\n PrecursorMass = 1458.725563 } PeptideSequence LIFQYASFNNSR RetentionTime 48.099266666667 Charge 2 LabeledMass 739.34337200254 PrecursorMass 1458.7255629902602 2 { PeptideSequence = "GILASDESNATTGK"\n RetentionTime = 23.07343333\n Charge = 2\n LabeledMass = 690.3148996\n PrecursorMass = 1362.662688 } PeptideSequence GILASDESNATTGK RetentionTime 23.073433333333 Charge 2 LabeledMass 690.3148995877 PrecursorMass 1362.6626879473802 3 { PeptideSequence = "FIESQVAK"\n RetentionTime = 22.022\n Charge = 2\n LabeledMass = 466.2408171\n PrecursorMass = 920.4967323 } PeptideSequence FIESQVAK RetentionTime 22.022 Charge 2 LabeledMass 466.24081707169 PrecursorMass 920.49673227576 4 { PeptideSequence = "SVVSIPHGPSIIAAR"\n RetentionTime = 43.30838333\n Charge = 2\n LabeledMass = 762.4060812\n PrecursorMass = 1502.856912 } PeptideSequence SVVSIPHGPSIIAAR RetentionTime 43.308383333333 Charge 2 LabeledMass 762.4060811573 PrecursorMass 1502.85691151298 5 { PeptideSequence = "TPLANLVYWK"\n RetentionTime = 54.7532\n Charge = 2\n LabeledMass = 609.3206006\n PrecursorMass = 1203.665195 } PeptideSequence TPLANLVYWK RetentionTime 54.7532 Charge 2 LabeledMass 609.320600565225 PrecursorMass 1203.66519458263 6 { PeptideSequence = "AVSLVLPSLK"\n RetentionTime = 50.83193333\n Charge = 2\n LabeledMass = 519.3152093\n PrecursorMass = 1025.648482 } PeptideSequence AVSLVLPSLK RetentionTime 50.831933333333 Charge 2 LabeledMass 519.315209325455 PrecursorMass 1025.64848188989 7 { PeptideSequence = "LSELLGKPVTK"\n RetentionTime = 31.73891667\n Charge = 2\n LabeledMass = 599.3468153\n PrecursorMass = 1183.717624 } PeptideSequence LSELLGKPVTK RetentionTime 31.738916666667 Charge 2 LabeledMass 599.346815312505 PrecursorMass 1183.7176240771903 8 { PeptideSequence = "LVDELNAGTIPR"\n RetentionTime = 27.41151667\n Charge = 3\n LabeledMass = 438.5593842\n PrecursorMass = 1296.703765 } PeptideSequence LVDELNAGTIPR RetentionTime 27.411516666667 Charge 3 LabeledMass 438.55938420378334 PrecursorMass 1296.7037649165195 9 { PeptideSequence = "MASMTGGQQMGR"\n RetentionTime = 46.23138333\n Charge = 2\n LabeledMass = 636.2478217\n PrecursorMass = 1253.531497 } PeptideSequence MASMTGGQQMGR RetentionTime 46.231383333333 Charge 2 LabeledMass 636.2478216939151 PrecursorMass 1253.53149726641 10 { PeptideSequence = "TFNDALADAK"\n RetentionTime = 28.61993333\n Charge = 2\n LabeledMass = 539.2464053\n PrecursorMass = 1064.513839 } PeptideSequence TFNDALADAK RetentionTime 28.619933333333 Charge 2 LabeledMass 539.2464052720301 PrecursorMass 1064.51383888964 11 { PeptideSequence = "VLNTWADIINR"\n RetentionTime = 51.82786667\n Charge = 2\n LabeledMass = 666.3366654\n PrecursorMass = 1313.709185 } PeptideSequence VLNTWADIINR RetentionTime 51.827866666667 Charge 2 LabeledMass 666.336665386335 PrecursorMass 1313.70918465125 12 { PeptideSequence = "NILLNEGIR"\n RetentionTime = 40.2474\n Charge = 2\n LabeledMass = 528.2854424\n PrecursorMass = 1040.597843 } PeptideSequence NILLNEGIR RetentionTime 40.2474 Charge 2 LabeledMass 528.28544237001 PrecursorMass 1040.5978432988 13 { PeptideSequence = "TWFDDADDWLR"\n RetentionTime = 57.13171667\n Charge = 2\n LabeledMass = 728.2912275\n PrecursorMass = 1438.615344 } PeptideSequence TWFDDADDWLR RetentionTime 57.131716666667 Charge 2 LabeledMass 728.29122753092 PrecursorMass 1438.61534383382 14 { PeptideSequence = "YWTMWK"\n RetentionTime = 48.49906667\n Charge = 2\n LabeledMass = 462.2017562\n PrecursorMass = 913.4156455 } PeptideSequence YWTMWK RetentionTime 48.499066666667 Charge 2 LabeledMass 462.2017562402751 PrecursorMass 913.41564550633 15 { PeptideSequence = "LQNIVGVPTSIR"\n Retent

The output of the function consensLookup is crystal clear: You have PeptideSequence, rt, charge, labledTheoMass and precursorMass. The next step is that we create a query to the database to extract the intensities of all peaks that are +/-5 min of our retention time and within 0.04 m/z of our peptide of interest. After we are done, we close the connection to the database. Dont confuse yourself with the helper function at the top, this is for a so-called baseline correction that is used for noise reduction. 

In [8]:
// Helper function to subtract an estimated baseline from intensity values.
// For very long traces (>500 points), skip correction and return original data.
let substractBaseLine (yData: float []) =
    if yData.Length > 500 then
        // No baseline correction for long signals (performance/robustness choice).
        yData
    else
        // Estimate baseline using asymmetric least squares (ALS).
        let baseLine =
            FSharp.Stats.Signal.Baseline.baselineAls 10 6 0.05 yData
            |> Array.ofSeq

        // Subtract baseline point-by-point and clamp negative values to zero.
        Array.map2 (fun y b ->
            let corrected = y - b
            if corrected < 0. then 0. else corrected
        ) yData baseLine

// the important part for you starts here
// Create retention-time query windows (center = consensus RT, tolerance = +/- 5).
let rtQuery = 
    consensLookup
    |> Array.map (fun lookup -> 
        MzIO.Processing.Query.createRangeQuery lookup.RetentionTime 5.)
// Create m/z query windows (center = labeled mass, tolerance = +/- 0.04).
let mzQuery = 
    consensLookup 
    |> Array.map (fun lookup -> 
        MzIO.Processing.Query.createRangeQuery lookup.LabeledMass 0.04
    )
// Pair each m/z query with its matching RT query (same peptide index).
let combine =  mzQuery  |> Array.zip rtQuery 
combine

index value 0 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 42.4451 LowValue 37.4451 HighValue 47.4451 Item2 MzIO.Processing.RangeQuery LockValue 657.360140482545 LowValue 657.3201404825451 HighValue 657.400140482545 1 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 48.099266666667 LowValue 43.099266666667 HighValue 53.099266666667 Item2 MzIO.Processing.RangeQuery LockValue 739.34337200254 LowValue 739.3033720025401 HighValue 739.38337200254 2 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 23.073433333333 LowValue 18.073433333333 HighValue 28.073433333333 Item2 MzIO.Processing.RangeQuery LockValue 690.3148995877 LowValue 690.2748995877 HighValue 690.3548995877 3 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 22.022 LowValue 17.022 HighValue 27.022 Item2 MzIO.Processing.RangeQuery LockValue 466.24081707169 LowValue 466.20081707169 HighValue 466.28081707169 4 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 43.308383333333 LowValue 38.308383333333 HighValue 48.308383333333 Item2 MzIO.Processing.RangeQuery LockValue 762.4060811573 LowValue 762.3660811573001 HighValue 762.4460811573 5 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 54.7532 LowValue 49.7532 HighValue 59.7532 Item2 MzIO.Processing.RangeQuery LockValue 609.320600565225 LowValue 609.2806005652251 HighValue 609.360600565225 6 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 50.831933333333 LowValue 45.831933333333 HighValue 55.831933333333 Item2 MzIO.Processing.RangeQuery LockValue 519.315209325455 LowValue 519.2752093254551 HighValue 519.355209325455 7 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 31.738916666667 LowValue 26.738916666667 HighValue 36.738916666666995 Item2 MzIO.Processing.RangeQuery LockValue 599.346815312505 LowValue 599.306815312505 HighValue 599.386815312505 8 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 27.411516666667 LowValue 22.411516666667 HighValue 32.411516666667 Item2 MzIO.Processing.RangeQuery LockValue 438.55938420378334 LowValue 438.5193842037833 HighValue 438.59938420378336 9 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 46.231383333333 LowValue 41.231383333333 HighValue 51.231383333333 Item2 MzIO.Processing.RangeQuery LockValue 636.2478216939151 LowValue 636.2078216939151 HighValue 636.287821693915 10 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 28.619933333333 LowValue 23.619933333333 HighValue 33.619933333333 Item2 MzIO.Processing.RangeQuery LockValue 539.2464052720301 LowValue 539.2064052720301 HighValue 539.28640527203 11 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 51.827866666667 LowValue 46.827866666667 HighValue 56.827866666667 Item2 MzIO.Processing.RangeQuery LockValue 666.336665386335 LowValue 666.296665386335 HighValue 666.3766653863349 12 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 40.2474 LowValue 35.2474 HighValue 45.2474 Item2 MzIO.Processing.RangeQuery LockValue 528.28544237001 LowValue 528.24544237001 HighValue 528.3254423700099 13 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 57.131716666667 LowValue 52.131716666667 HighValue 62.131716666667 Item2 MzIO.Processing.RangeQuery LockValue 728.29122753092 LowValue 728.25122753092 HighValue 728.3312275309199 14 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 48.499066666667 LowValue 43.4990

The output here gives you a tuple of rt and mass query, which is the input for the function which creates the XICs. Dont worry if this takes some minutes.

In [9]:
// Build one XIC for each (retention time, m/z) pair from `combine`
let xics = 
    combine 
    |> Array.map (fun (rt, mz) -> 
        MzIO.Processing.Query.getXIC mzReader idx rt mz)
// Baseline-correct each XIC trace
let baselineCorrection = 
    xics
    |> Array.map (fun xic ->
        let intensity,mz,rt =
        // Split the XIC points into separate arrays
            xic |> Array.map (fun x -> x.Intensity),
            xic |> Array.map (fun x -> x.Mz),
            xic |> Array.map (fun x -> x.Rt)
        // Pair RT with baseline-corrected intensity
        Array.zip rt (substractBaseLine intensity)
    )

transaction.Dispose()

// Keep corrected traces together with consensus lookup data
let CorrAndRecord =  consensLookup |> Array.zip baselineCorrection

// Create one point chart per corrected trace
let xicChart =
    CorrAndRecord 
    |> Array.map (fun x -> 
        fst x
        |> Chart.Point 
        |> Chart.withXAxisStyle "Retention Time"
        |> Chart.withYAxisStyle "Intensity/Score"
        |> Chart.withSize (900.,900.)
    )
xicChart

index,value
0,<!-- Plotly chart will be drawn inside this DIV -->
1,<!-- Plotly chart will be drawn inside this DIV -->
2,<!-- Plotly chart will be drawn inside this DIV -->
3,<!-- Plotly chart will be drawn inside this DIV -->
4,<!-- Plotly chart will be drawn inside this DIV -->
5,<!-- Plotly chart will be drawn inside this DIV -->
6,<!-- Plotly chart will be drawn inside this DIV -->
7,<!-- Plotly chart will be drawn inside this DIV -->
8,<!-- Plotly chart will be drawn inside this DIV -->
9,<!-- Plotly chart will be drawn inside this DIV -->


### Hip Hip Hoorey!
We have our XICs (slay) Now we use the second derivative to identify peaks with our trace

In [10]:
// Detect peaks on each corrected XIC and keep its associated record info
let peaks = 
    CorrAndRecord
    |> Array.map (fun peak ->
        // peak is expected as: (correctedTrace, recordInfo)
        fst peak
        // correctedTrace is (rt, intensity)[] -> split into separate arrays
        |> Array.unzip
        |> (fun (ret, intensity) ->
            // Second-derivative peak detection on RT/intensity signal
            // args: threshold, minDistance, smoothingWindow
            FSharp.Stats.Signal.PeakDetection.SecondDerivative.getPeaks 0.1 2 13 ret intensity
        ), snd peak
    )
peaks |> Array.head

(FSharp.Stats.Signal.PeakDetection+IdentifiedPeak[], { PeptideSequence = "LQNIVGVPTSIR"\n RetentionTime = 42.4451\n Charge = 2\n LabeledMass = 657.3601405\n PrecursorMass = 1295.756135 }) Item1 index value 0 { Apex = { Index = 4\n XVal = 37.5742\n YVal = 20341.91933 }\n LeftLiftOff = None\n LeftEnd = { Index = 0\n XVal = 37.45886667\n YVal = 3331.91608 }\n RightLiftOff = Some { Index = 11\n XVal = 37.77568333\n ... Apex { Index = 4\n XVal = 37.5742\n YVal = 20341.91933 } Index 4 XVal 37.5742 YVal 20341.919332187175 LeftLiftOff <null> LeftEnd { Index = 0\n XVal = 37.45886667\n YVal = 3331.91608 } Index 0 XVal 37.458866666667 YVal 3331.916079917577 RightLiftOff Some({ Index = 11\n XVal = 37.77568333\n YVal = 1980.147762 }) Value { Index = 11\n XVal = 37.77568333\n YVal = 1980.147762 } Index 11 XVal 37.775683333333 YVal 1980.1477620434807 RightEnd { Index = 12\n XVal = 37.80451667\n YVal = 2058.223599 } Index 12 XVal 37.804516666667 YVal 2058.2235988103484 LeftSidedConvolved True RightSidedConvolved True XData [ 37.458866666667, 37.4877, 37.5167, 37.545533333333, 37.5742, 37.603033333333, 37.6317, 37.660366666667, 37.6892, 37.718016666667, 37.74685, 37.775683333333, 37.804516666667 ] YData [ 3331.916079917577, 7327.728269239101, 12545.110313338953, 16601.59921855144, 20341.919332187175, 18773.159146009853, 15176.58423922001, 11296.60737543185, 7100.310154588832, 3922.284190810522, 2535.2595360649266, 1980.1477620434807, 2058.2235988103484 ] 1 { Apex = { Index = 16\n XVal = 37.91951667\n YVal = 4340.599129 }\n LeftLiftOff = Some { Index = 11\n XVal = 37.77568333\n YVal = 1980.147762 }\n LeftEnd = { Index = 12\n XVal = 37.80451667\n YVal = 2058.2235... Apex { Index = 16\n XVal = 37.91951667\n YVal = 4340.599129 } Index 16 XVal 37.919516666667 YVal 4340.599128825231 LeftLiftOff Some({ Index = 11\n XVal = 37.77568333\n YVal = 1980.147762 }) Value { Index = 11\n XVal = 37.77568333\n YVal = 1980.147762 } Index 11 XVal 37.775683333333 YVal 1980.1477620434807 LeftEnd { Index = 12\n XVal = 37.80451667\n YVal = 2058.223599 } Index 12 XVal 37.804516666667 YVal 2058.2235988103484 RightLiftOff Some({ Index = 21\n XVal = 38.063\n YVal = 0.0 }) Value { Index = 21\n XVal = 38.063\n YVal = 0.0 } Index 21 XVal 38.063 YVal 0 RightEnd { Index = 21\n XVal = 38.063\n YVal = 0.0 } Index 21 XVal 38.063 YVal 0 LeftSidedConvolved True RightSidedConvolved True XData [ 37.804516666667, 37.83335, 37.862016666667, 37.89085, 37.919516666667, 37.948183333333, 37.976833333333, 38.0055, 38.034166666667, 38.063 ] YData [ 2058.2235988103484, 3916.6184938979623, 4358.63450518466, 5006.482797156338, 4340.599128825231, 3089.4720764813947, 938.61783834371, 0, 0, 0 ] 2 { Apex = { Index = 28\n XVal = 38.26531667\n YVal = 32554.28383 }\n LeftLiftOff = Some { Index = 21\n XVal = 38.063\n YVal = 0.0 }\n LeftEnd = { Index = 21\n XVal = 38.063\n YVal = 0.0 }\n RightLiftOff = Som... Apex { Index = 28\n XVal = 38.26531667\n YVal = 32554.28383 } Index 28 XVal 38.265316666667 YVal 32554.283827392606 LeftLiftOff Some({ Index = 21\n XVal = 38.063\n YVal = 0.0 }) Value { Index = 21\n XVal = 38.063\n YVal = 0.0 } Index 21 XVal 38.063 YVal 0 LeftEnd { Index = 21\n XVal = 38.063\n YVal = 0.0 } Index 21 XVal 38.063 YVal 0 RightLiftOff Some({ Index = 36\n XVal = 38.49481667\n YVal = 2274.449625 }) Value { Index = 36\n XVal = 38.49481667\n YVal = 2274.449625 } Index 36 XVal 38.494816666667 YVal 2274.4496245131454 RightEnd { Index = 43\n XVal = 38.69713333\n YVal = 0.0 } Index 43 XVal 38.697133333333 YVal 0 LeftSidedConvolved True RightSidedConvolved True XData [ 38.063, 38.091833333333, 38.120666666667, 38.149666666667, 38.1785, 38.2075, 38.2365, 38.265316666667, 38.29415, 38.322983333333, 38.35165, 38.380316666667, 38.408816666667, 38.437483333333, 38.46615, 38.494816666667, 38.523633333333, 38.552466666667, 38.581483333333, 38.610466666667 ... (3 more) ] YData [ 0, 160.79135728369056, 687.5384120973549, 4562.9342120954625, 12277.403804270773, 20272.966420740668, 27387.46929934164, 32554

The peak model includes numerus information. Therefore we can mark the apices of the peaks we identified.


In [11]:
// Code-Block 4
// Extract only apex coordinates (RT, intensity) from each detected subpeak
// while preserving associated metadata from `peaks`
let apices =
    peaks
    |> Array.map (fun peak ->
        fst peak
        |> Array.map (fun subpeak -> subpeak.Apex.XVal,subpeak.Apex.YVal), snd peak
    )

// For each trace: overlay detected apex points on top of baseline-corrected XIC
let apicesChart=
    Array.map2 (fun apice baseLineC ->
        [    
            Chart.Point(apice, Name = "apices")
            |> Chart.withMarkerStyle(Size = 15)
            Chart.Point(baseLineC, Name = "XIC")

        ]
        |> Chart.combine
        |> Chart.withXAxisStyle "Retention Time"
        |> Chart.withYAxisStyle "Intensity"
        |> Chart.withSize (900., 900.)
    ) (apices |> Array.map (fun (x,y) -> x)) baselineCorrection
apicesChart


index,value
0,<!-- Plotly chart will be drawn inside this DIV -->
1,<!-- Plotly chart will be drawn inside this DIV -->
2,<!-- Plotly chart will be drawn inside this DIV -->
3,<!-- Plotly chart will be drawn inside this DIV -->
4,<!-- Plotly chart will be drawn inside this DIV -->
5,<!-- Plotly chart will be drawn inside this DIV -->
6,<!-- Plotly chart will be drawn inside this DIV -->
7,<!-- Plotly chart will be drawn inside this DIV -->
8,<!-- Plotly chart will be drawn inside this DIV -->
9,<!-- Plotly chart will be drawn inside this DIV -->


We can then go ahead and characterize our peaks and quantify the area under the fitted curve.


In [12]:
// Code-Block 5
// Quantify one peak per trace using the RT of the most intense apex
let quantifiedXIC = 
    Array.map2 (fun rt (peak, metaData) -> 
        // Pick the detected peak at/near the target RT
        BioFSharp.Mz.Quantification.HULQ.getPeakBy peak rt
        // quantify peak of interest
        |> BioFSharp.Mz.Quantification.HULQ.quantifyPeak,
        metaData
    ) (apices |> Array.map (fun a -> a |> fst |> Array.maxBy snd |> fst)) peaks

// Extract only the quantified peak area values
let area = 
    quantifiedXIC
    |> Array.map (fun (x, metaData) ->  x.Area)
area


[ 5797.112015844433, 1568.4906661554894, 2857.0835572126634, 4527.288639034969, 1615.0034338014473, 2253.3420625423714, 4529.904159038975, 1671.3970755838568, 552.9358225956541, 97.86057823444033, 6066.423899760952, 1590.949692543897, 6351.759960542036, 1401.201646658353, 2046.711108756307, 828.1404307127499, 1740.992048629933, 1583.5774805028288, 16619.104394508195, 1446.8889352401304 ... (105 more) ]

The peak model gives us all the information we need for our peptide of interest. If we want to see what we quantified , we can take an 
exponential modified gaussian function (lol) using the parameters given by the peak model and plot it together with the previously extracted XIC.


In [13]:
open BioFSharp.Mz
// Code-Block 6
// For each quantified peak, build an evaluable fitted-curve function (if a model exists)
// and keep the corresponding metadata attached.
let evaluation =
    quantifiedXIC
    |> Array.map (fun (peak, metaData) ->
        match peak.Model with
        // EMG fit: create function value evaluator from estimated EMG parameters
        | Some (Quantification.HULQ.PeakModel.EMG model) ->
            (Fitting.NonLinearRegression.Table.emgModel.GetFunctionValue (vector peak.EstimatedParams), metaData)
            |> Some
        // Gaussian fit: create function value evaluator from estimated Gaussian parameters
        | Some (Quantification.HULQ.PeakModel.Gaussian model) ->
            (Fitting.NonLinearRegression.Table.gaussModel.GetFunctionValue (vector peak.EstimatedParams), metaData)
            |> Some
        | None -> None
    )
        

In [14]:
// Code-Block 7
// Evaluate fitted model values on baseline-corrected RT points, keeping metadata
let quantifiedArea =
    Array.map2 (fun p (e: ((float -> float)*ConsensLookup) option) ->
        if e.IsSome then
            (
            // For each (rt, intensity) point in baseline-corrected trace:
            // keep rt and replace intensity with model-predicted intensity
                p
                |> Array.map (fun (x, _) ->
                    x, fst(e.Value) x
                ),
                snd(e.Value)
            )
            |> Some
        else None
    ) baselineCorrection evaluation


let quantifiedAreaChart =
    Array.map2 (fun baselineC (qA: option<array<float * float>* ConsensLookup>) ->
        if qA.IsSome then
            let data,metadata = qA.Value
            [
                Chart.Point(baselineC, Name="XIC" )
                Chart.SplineArea(data, Name = "quantified XIC")
            ]
            |> Chart.combine
            |> Chart.withTitle($"Peptide:{metadata.PeptideSequence} Charge:{metadata.Charge} Retentiontime:{metadata.RetentionTime}")
            |> Chart.withXAxisStyle (TitleText = "Retention Time")
            |> Chart.withYAxisStyle "Intensity"
            |> Chart.withSize (900., 900.)
            |> Some
        else None
    ) baselineCorrection quantifiedArea
    |> Array.choose id

quantifiedAreaChart

index value 0 <!-- Plotly chart will be drawn inside this DIV --> 1 <!-- Plotly chart will be drawn inside this DIV --> 2 <!-- Plotly chart will be drawn inside this DIV --> 3 <!-- Plotly chart will be drawn inside this DIV --> 4 <!-- Plotly chart will be drawn inside this DIV --> 5 <!-- Plotly chart will be drawn inside this DIV --> 6 <!-- Plotly chart will be drawn inside this DIV -->

### Wonderful! 
The result is speaking for itself :) The final step is to get you another .tsv file were all important data is stored.

In [15]:
let output = 
    consensLookup
    |> FSharpAux.IO.SeqIO.Seq.CSV "\t" true false
    |> fun csv -> System.IO.File.WriteAllLines("/home/paulinehans/Dokumente/relativeQuantification.tsv", csv)
output

## Questions
1. Why is a retention-time index from MS1 needed when extracting XICs from MS2-related queries?
2. What is the purpose of grouping by peptide sequence and charge before building consensLookup?
3. How might your quantification change if you choose a different rule than “highest matching score” within each group?
4. Why can baseline correction improve peak detection and downstream quantification?
5. Which signal characteristics does second-derivative peak detection emphasize, and when can it fail?
6. How does selecting the apex with highest intensity influence which peak is quantified?
7. How would you evaluate whether the fitted curve is biologically meaningful and not just mathematically good?



